# DSPy ChainOfThought — One-Shot Example

`dspy.ChainOfThought` wraps a signature and automatically inserts a `reasoning` step before the declared outputs.
The model must explain its thinking before committing to an answer.

Providing a **one-shot example** teaches the model *how* to reason — the structure and style to follow.

This demo shows:
1. ChainOfThought with no example (zero-shot)
2. ChainOfThought with a single example (one-shot)
3. How to inspect the prompt to see the example injected

In [1]:
# Dependencies:
# !pip install dspy python-dotenv

In [2]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

lm = dspy.LM(
    model="anthropic/claude-haiku-4.5",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api",
    extra_headers={
        "HTTP-Referer": "https://my-app.com",
        "X-Title": "DSPy CoT Demo",
    },
)
dspy.configure(lm=lm)

## 1. Define the Task

A `Signature` declares typed inputs and outputs.
`ChainOfThought` wraps it and inserts a `reasoning` field before the declared outputs.

The augmented signature becomes: `review → reasoning, sentiment`

In [3]:
class SentimentClassifier(dspy.Signature):
    """Classify the sentiment of a product review."""
    review: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")

cot = dspy.ChainOfThought(SentimentClassifier)

# The inner predictor's signature shows the reasoning field added by ChainOfThought
print(cot.predict.signature)

StringSignature(review -> reasoning, sentiment
    instructions='Classify the sentiment of a product review.'
    review = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Review:', 'desc': '${review}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'desc': '${reasoning}', '__dspy_field_type': 'output', 'prefix': 'Reasoning:'})
    sentiment = Field(annotation=str required=True json_schema_extra={'desc': 'positive, negative, or neutral', '__dspy_field_type': 'output', 'prefix': 'Sentiment:'})
)


## 2. Zero-Shot: No Example

The model reasons and classifies from scratch — no guidance on reasoning style.

In [4]:
result = cot(review="The battery life is decent but the screen scratches way too easily.")

print("Reasoning:", result.reasoning)
print("Sentiment:", result.sentiment)

Reasoning: This review contains both positive and negative elements. The user acknowledges that battery life is "decent" (positive aspect), but expresses a significant complaint about the screen scratching easily (negative aspect). The negative complaint appears to be the primary concern and is stated more emphatically, suggesting dissatisfaction with a key product feature. Overall, the review leans toward negative despite the acknowledgment of one acceptable feature.
Sentiment: negative


## 3. One-Shot: Provide a Single Example

A `dspy.Example` passed as a demo teaches the model the reasoning format to follow.
The example must include all fields of the augmented signature: `review`, `reasoning`, `sentiment`.

In DSPy, demos live on the inner predictor: `cot.predict.demos`.

In [5]:
cot.predict.demos = [
    dspy.Example(
        review="Absolutely love this keyboard — quiet switches, great build quality, worth every penny.",
        reasoning=(
            "Key signals: 'absolutely love' (strong approval), 'great build quality' (positive attribute), "
            "'worth every penny' (satisfaction with value). "
            "No negative qualifiers. Tone is enthusiastic throughout."
        ),
        sentiment="positive",
    ).with_inputs("review")
]

In [6]:
# Same review as before — observe how the reasoning style now mirrors the example
result = cot(review="The battery life is decent but the screen scratches way too easily.")

print("Reasoning:", result.reasoning)
print("Sentiment:", result.sentiment)

Reasoning: Mixed signals present: 'decent' for battery life is mildly positive, but 'scratches way too easily' is a significant negative complaint about durability. The negative aspect addresses a fundamental product quality issue, which outweighs the acceptable battery performance.
Sentiment: negative


## 4. Inspect the Prompt

`dspy.inspect_history()` shows the exact messages sent to the model.
The one-shot example appears as a completed assistant turn before the actual question.

In [7]:
dspy.inspect_history(n=1)





[2026-06-22T12:47:06.140200]

System message:

Your input fields are:
1. `review` (str):
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (str): positive, negative, or neutral
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## review ## ]]
{review}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a product review.


User message:

[[ ## review ## ]]
Absolutely love this keyboard — quiet switches, great build quality, worth every penny.


Assistant message:

[[ ## reasoning ## ]]
Key signals: 'absolutely love' (strong approval), 'great build quality' (positive attribute), 'worth every penny' (satisfaction with value). No negative qualifiers. Tone is enthusiastic throughout.

[[ ## sentiment ## ]]
positive

[[ ## completed ## ]]


User message:

[[ ## review ## ]]
The battery life is decent b

## Summary

| | Zero-shot CoT | One-shot CoT |
|---|---|---|
| Reasoning style | Model chooses freely | Follows the example's structure |
| Output format | May vary across calls | Consistent with the demo |
| Extra cost | None | +example tokens per call |
| When to use | Any task | When you need a specific reasoning style or format |

**Where the demo lives:** `cot.predict.demos` — a list, so multiple examples (few-shot) work the same way.

**Fields required in the example:** must match the augmented signature — inputs plus all output fields including `reasoning`.